# 🎤 AI Vocals Studio — Pacaveli Full Clone Pipeline

**BEFORE RUNNING:**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Upload `Pacaveli_vocals_training.zip` to your Google Drive
3. Click **Runtime → Run All** (Ctrl+F9)

**What this does:**
- Strips music from vocals using Demucs (pure voice only)
- Preprocesses audio to 22kHz SO-VITS format
- Extracts HuBERT speaker features (~30 min on GPU)
- Trains the voice model (~8-10 hrs)
- Auto-backup to Drive every 30 min
- Downloads trained model automatically

In [ ]:
# ── Step 1: Check GPU ────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────
!pip install -q so-vits-svc-fork==4.2.30 torchcodec demucs
print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Mount Google Drive ───────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/ai_vocals_checkpoints'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'✅ Drive mounted. Backups → {BACKUP_DIR}')

In [ ]:
# ── Step 4: Find + extract training zip ──────────────────────────
import glob, zipfile, os

zip_path = None
for pat in ['/content/drive/MyDrive/**/*vocals*.zip',
             '/content/drive/MyDrive/**/*Pacaveli*.zip',
             '/content/drive/MyDrive/**/*training*.zip',
             '/content/drive/MyDrive/**/*.zip']:
    found = glob.glob(pat, recursive=True)
    if found:
        zip_path = max(found, key=os.path.getmtime)
        print(f'✅ Found zip: {os.path.basename(zip_path)}')
        break

if not zip_path:
    print('No zip in Drive — upload now:')
    from google.colab import files
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]

print(f'📦 Extracting...')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/')

raw = glob.glob('/content/raw_vocals/*.*')
print(f'✅ {len(raw)} raw audio files ready')
for f in raw: print(f'   {os.path.basename(f)}')

In [ ]:
# ── Step 5: Demucs — isolate pure vocals, strip any music ────────
import glob, os

raw_files = (glob.glob('/content/raw_vocals/*.mp3') +
             glob.glob('/content/raw_vocals/*.wav') +
             glob.glob('/content/raw_vocals/*.flac') +
             glob.glob('/content/raw_vocals/*.m4a'))

print(f'🎵 Demucs: isolating vocals from {len(raw_files)} files (~2-5 min on GPU)...')
os.makedirs('/content/demucs_out', exist_ok=True)

for audio_file in sorted(raw_files):
    fname = os.path.basename(audio_file)
    print(f'   → {fname}')
    os.system(f'demucs --two-stems=vocals --mp3 -n mdx_extra -o /content/demucs_out "{audio_file}"')

vocal_files = glob.glob('/content/demucs_out/**/vocals.mp3', recursive=True)
print(f'\n✅ Extracted {len(vocal_files)} clean vocal tracks')

# Move into dataset/Pacaveli/ as WAV
os.makedirs('/content/dataset/Pacaveli', exist_ok=True)
for i, vf in enumerate(sorted(vocal_files), 1):
    dst = f'/content/dataset/Pacaveli/speaker_{i:04d}.wav'
    os.system(f'ffmpeg -y -i "{vf}" -ar 44100 "{dst}" -loglevel quiet')
    print(f'   speaker_{i:04d}.wav')

print('\n✅ Clean vocals ready in /content/dataset/Pacaveli/')

In [ ]:
# ── Step 6: svc pre-config (resample to 22kHz + generate filelists)
import os, shutil, glob
os.chdir('/content')

# Copy base model checkpoints if included in zip
os.makedirs('/content/logs/44k', exist_ok=True)
for ckpt in ['G_0.pth', 'D_0.pth']:
    src = f'/content/base_model/{ckpt}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/logs/44k/{ckpt}')
        print(f'   Copied base checkpoint: {ckpt}')

print('🔧 Running svc pre-config...')
os.system('svc pre-config -t so-vits-svc-4.0v1')

wavs = glob.glob('/content/dataset/44k/**/*.wav', recursive=True)
print(f'✅ Pre-config done — {len(wavs)} resampled audio files')

In [ ]:
# ── Step 7: Extract HuBERT features (GPU, ~20-40 min) ────────────
import os, glob
os.chdir('/content')

print('🧠 Extracting HuBERT speaker features...')
print('   ~20-40 min on T4 GPU (was 8.5 hrs on CPU)')
os.system('svc pre-hubert')

features = glob.glob('/content/dataset/44k/**/*.data.pt', recursive=True)
print(f'\n✅ Features extracted: {len(features)} files')
if not features:
    raise RuntimeError('HuBERT extraction failed — check output above')

In [ ]:
# ── Step 8: Resume from previous checkpoint (if any) ─────────────
import glob, shutil, os

BACKUP_DIR = '/content/drive/MyDrive/ai_vocals_checkpoints'
os.makedirs('/content/logs/44k', exist_ok=True)

existing_G = sorted(
    glob.glob(f'{BACKUP_DIR}/G_*.pth'),
    key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p))) or '0')
)

if existing_G:
    latest_G = existing_G[-1]
    latest_D = latest_G.replace('G_', 'D_')
    epoch = ''.join(filter(str.isdigit, os.path.basename(latest_G)))
    print(f'🔄 Resuming from epoch {epoch}: {os.path.basename(latest_G)}')
    shutil.copy(latest_G, '/content/logs/44k/')
    if os.path.exists(latest_D):
        shutil.copy(latest_D, '/content/logs/44k/')
        print(f'   Discriminator: {os.path.basename(latest_D)}')
else:
    print('🆕 No previous checkpoints — starting from scratch')

In [ ]:
# ── Step 9: Auto-backup thread (every 30 min → Drive) ────────────
import threading, time, glob, shutil, os

BACKUP_DIR = '/content/drive/MyDrive/ai_vocals_checkpoints'

def backup_loop():
    while True:
        time.sleep(1800)
        ckpts = glob.glob('/content/logs/44k/G_*.pth')
        if ckpts:
            for c in ckpts:
                shutil.copy(c, os.path.join(BACKUP_DIR, os.path.basename(c)))
            for d in glob.glob('/content/logs/44k/D_*.pth'):
                shutil.copy(d, os.path.join(BACKUP_DIR, os.path.basename(d)))
            print(f'💾 Backed up {len(ckpts)} checkpoints to Drive')

threading.Thread(target=backup_loop, daemon=True).start()
print('✅ Auto-backup active — checkpoints saved to Drive every 30 min')

In [ ]:
# ── Step 10: TRAIN ───────────────────────────────────────────────
import os
os.chdir('/content')

print('🚀 TRAINING: Pacaveli Voice Clone')
print('   Source: clean acapella vocals (demucs-separated)')
print('   Checkpoints saved every 100 epochs')
print('   Drive backup every 30 minutes')
print('   Est. time: 8-10 hours on T4 GPU')
print('')

os.system('svc train -c configs/44k/config.json -m logs/44k')

print('\n✅ Training complete!')

In [ ]:
# ── Step 11: Final backup + download trained model ────────────────
import glob, os, shutil
from google.colab import files

BACKUP_DIR = '/content/drive/MyDrive/ai_vocals_checkpoints'

ckpts = sorted(glob.glob('/content/logs/44k/G_*.pth'),
    key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p))) or '0'))

print('All checkpoints:', [os.path.basename(c) for c in ckpts])

if ckpts:
    best = ckpts[-1]
    # Final Drive backup
    for c in ckpts:
        shutil.copy(c, BACKUP_DIR)
        d = c.replace('G_', 'D_')
        if os.path.exists(d): shutil.copy(d, BACKUP_DIR)
    shutil.copy('/content/logs/44k/config.json', BACKUP_DIR)
    print(f'✅ All backed up to Drive: {BACKUP_DIR}')
    # Download
    print(f'\n📥 Downloading {os.path.basename(best)} + config.json')
    files.download(best)
    files.download('/content/logs/44k/config.json')
    print('\n🎉 DONE! Move to models/Pacaveli/ then click Install Model in the app.')
else:
    print('⚠️  No checkpoints — check training log above')